# 260514 Self RAG 구현 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w9_agent_rag/llm_260514_self_rag_1.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: Self-RAG 핵심 개념

- **Adaptive-RAG vs Self-RAG**: 어댑티브는 "쿼리 보고 검색할지" 1군데만 판단 → Self-RAG는 "검색된 문서 + 생성된 답변"까지 LLM이 스스로 검열. 평가 지점이 1개 → 3개로 늘어남.
- **검열 3단계 (reflection token)**: ① IsRelevant(문서↔질문 연관?) ② IsSupported(답변이 문서 근거? = 환각 여부) ③ IsUseful(답변이 질문에 진짜 답하는지 1~5점). 원논문은 학습된 토큰, 구현체는 LLM-as-judge로 흉내.
- **State가 비대해지는 이유**: documents/filtered_documents/relevance_grades/hallucination_score/usefulness_score + retries 카운터들까지 — 평가 결과와 재시도 횟수를 다 들고 다녀야 분기 가능.
- **with_structured_output이 필수**: 평가 LLM에 "yes/no만" 시켜도 "yes입니다.", "no." 처럼 흔들림 → 분기 불가. `_Relevance(binary: Literal['yes','no'])` Pydantic 스키마로 묶어야 router 안정.
- **무한 루프 실증 (제로콜라 가격 예시)**: 도메인 밖 질문 → 관련성 다 `no` → filtered 빔 → retrieve로 회귀 → 같은 결과 무한 반복. `relevance_retries` + `max_tries` 가드 필수.
- **C-RAG의 쿼리 재작성을 빌려옴**: Self-RAG 원논문엔 없음. Corrective-RAG에서 가져와 "관련 문서 없음 → rewrite_query → retrieve"로 폴백. 동의어/구체화로 검색 살리기.
- **retrieve_aware 트릭**: `state['rewritten_question'] or state['question']` — 처음엔 원문, 한 바퀴 돌면 재작성문 사용. 노드 이름 유지하면서 내부만 갈아끼면 엣지 재배선 불필요.


## 강의 메모: 실무 팁 (Self-RAG 구현 함정)

- **"땜빵 vs 진짜 해결"**: 관련 문서 못 찾을 때 ① 빈 컨텍스트로 그냥 generate (포기/땜빵) ② 쿼리 재작성 후 재검색 (C-RAG). ②가 정공법이지만 도메인 밖 질문은 결국 ①로 막아야 함 → 재작성도 1회로 제한.
- **카운터 위치 주의**: `relevance_retries`는 `rewrite_query_node`에서 `+1` (재작성 횟수를 셈). 라우터에선 `if relevance_retries >= 1: return 'generate'`로 강제 탈출시켜야 무한 루프 차단.
- **환각 평가 프롬프트는 엄격하게**: 기본 프롬프트론 LLM이 일반 지식 섞인 답변도 `no(환각 아님)`로 봐줌. → "모든 사실이 문서에 명시적으로 있어야 no, 문서 밖 정보면 yes"식 엄격 채점관 페르소나 필수.
- **State 오타가 침묵 버그 (어제 사례)**: TypedDict 키 오타(`document` vs `documents`, sub_query 업데이트 누락)나도 그래프는 컴파일/실행됨. 결과만 빔 → 노드 return dict 키와 State 스키마 1:1 매칭 점검.
- **k값·컨텍스트 전략은 별개 레이어**: 검색 k=3→5, 하이브리드 서치+리랭킹은 retrieve 노드 안에서 처리. Self-RAG 평가 루프와 분리해야 디버깅 쉬움. Pydantic `UserWarning`은 정상이니 무시.
